# 抖音直播数据处理（合并优化版）

把原来的两个 notebook 合并成一份，**一次运行同时产出两个汇总文件**：

| 输出文件 | 口径 | 来源 |
| --- | --- | --- |
| `直播汇总.xlsx` | 全量订单（含未签收、在途） | 原「标准版」 |
| `直播汇总-到货订单.xlsx` | 只统计**已签收**订单（到货） | 原「到货订单」版 |

## 两个版本的差异 & 合并方式

| 差异点 | 标准版 | 到货订单版 | 合并处理 |
| --- | --- | --- | --- |
| 销售金额口径 | 订单应付金额 + 平台实际承担优惠 + 运费 | 商品数量 × 商品单价 + 运费 | 两列都算，各自取用：`销售金额` / `到货销售金额` |
| 包裹状态映射 | 已签收 / 派送中 / 待取件 → 已完成1 | 仅「已签收」→ 已完成1 | 先保存 `包裹状态_原始` 列，再统一映射；到货口径用原始状态 == 已签收 过滤 |
| 统计范围 | 全部订单 | 仅到货订单 | 同一条管线，最后按口径切分 |
| 指标列 | 含发货时长 / 揽件时长 / 站内订单占比 / 对标支付签收率等 | 不含 | 参数 `full_metrics` 控制 |
| 复购率公式 | (总支付单量 − 转化数) / 转化数 | 复购 / 转化数 | 参数控制 |

优化点：逐行 `for` 循环全部向量化、链式 `inplace` 改为显式赋值（pandas 3 下链式 inplace 实际不生效）、路径与列清单常量化、兼容 Python 3.7+ / 旧版 pandas。

In [1]:
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

# ---- 金额 / 数量列 ---------------------------------------------------------
NUMERIC_COLUMNS = [
    "商品数量", "商品单价", "运费", "订单应付金额", "平台实际承担优惠金额",
    "商家实际承担优惠金额", "订单实际收款金额", "运费险预计投保费用",
]


def to_number(series, name=""):
    """把金额 / 数量列统一转成数值。

    兼容导出 CSV 带千分位逗号、货币符号、或整列被读成文本的情况
    （旧版 pandas 会把这类列识别为 object，直接相加会报
    TypeError: can only concatenate str (not "float") to str）。
    """
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = series.astype(str)
    for token in (",", "¥", "￥", " ", "\t"):
        cleaned = cleaned.str.replace(token, "", regex=False)
    cleaned = cleaned.str.strip()
    converted = pd.to_numeric(cleaned, errors="coerce")
    lost = int(series.notna().sum() - converted.notna().sum())
    if lost:
        print("提示：%s 有 %d 个值无法解析为数字，已置空" % (name or "某列", lost))
    return converted


# ---- 文件路径 -------------------------------------------------------------
ORDER_CSV = "./1.csv"                 # 订单明细
PACKAGE_XLSX = "./2.xlsx"             # 包裹状态（运单）
SKU_XLSX = "./商品编码.xlsx"           # 商品编码 → 产品名称
CONSUME_XLSX = "./店铺消耗.xlsx"       # 广告消耗

SUMMARY_FILE = "直播汇总.xlsx"                # 全量汇总
SUMMARY_ARRIVAL_FILE = "直播汇总-到货订单.xlsx"  # 到货汇总
LACK_FILE = "缺码表.xlsx"                      # 缺码表
CHECK_FILE = "检查.xlsx"                       # 中间结果（排查用）
WRITE_CHECK = True                             # 不需要中间结果时改成 False

# ---- 达人白名单（两个版本一致，仅顺序不同） -------------------------------
INFLUENCER_IDS = [
    133518108394809, 573283167910355, 3300093554065689, 1224185856927387,
    68077694158, 2429274046346537, 2722825191308558, 0,
]

# ---- 包裹状态映射 & 到货口径 ---------------------------------------------
PACKAGE_TO_DONE = {"已签收": "已完成1", "派送中": "已完成1", "待取件": "已完成1"}
ARRIVAL_RAW_STATUS = "已签收"

# ---- 售后状态分组 ---------------------------------------------------------
DONE_AFTER_SALE = {"换货成功", "-", "售后关闭", "补寄/换货成功", "补寄成功"}
AFTER_SALE_IN_PROGRESS_EXCLUDE = {"换货成功", "-", "售后关闭", "退款成功"}

# ---- 透视 / 输出列定义 ----------------------------------------------------
BASE_PIVOT_VALUES = [
    "销售金额", "订单数", "在线支付已完成", "售后中", "退回成功", "已发货",
    "复购", "广告", "总支付单量", "售前退款单量", "备货单量", "揽件前退款单量", "揽件订单数",
]
EXTRA_PIVOT_VALUES = ["发货时长（小时）", "揽件时长（小时）", "站内订单", "优惠券单量"]

BASE_RESULT_COLUMNS = [
    "订单提交时间", "订单数", "销售金额", "售后中", "在线支付已完成", "已发货",
    "退回成功", "复购", "广告", "总支付单量", "售前退款单量", "备货单量",
    "揽件前退款单量", "揽件订单数",
]
EXTRA_RESULT_COLUMNS = BASE_RESULT_COLUMNS + EXTRA_PIVOT_VALUES

RENAME_COLUMNS = {
    "转化数": "广告单量",
    "订单数": "总发出单量",
    "订单提交时间": "日期",
    "销售金额": "客单",
}

FULL_SUMMARY_COLUMNS = [
    "日期", "产品", "广告单量", "总发出单量", "真实发货率", "总支付单量", "售前退款率",
    "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功", "签收率", "退回率",
    "复购率", "消耗", "发货时长", "揽件时长", "优惠订单占比", "站内订单占比", "对标支付签收率",
    "对标揽件单量签收率", "对标揽件单量退回率", "对标揽件单量复购率", "平均客单", "客单",
    "复购", "备货单量", "售前退款单量", "揽件前退款单量", "发货时长（小时）",
    "揽件时长（小时）", "站内订单", "优惠券单量",
]
FULL_ADVERT_COLUMNS = [
    "日期", "产品", "广告单量", "总发出单量", "真实发货率", "总支付单量", "售前退款率",
    "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功", "签收率", "退回率",
    "消耗", "发货时长", "揽件时长", "优惠订单占比", "站内订单占比", "对标支付签收率",
    "对标揽件单量签收率", "对标揽件单量退回率", "平均客单", "客单", "备货单量",
    "售前退款单量", "揽件前退款单量", "发货时长（小时）", "揽件时长（小时）",
    "站内订单", "优惠券单量",
]
ARRIVAL_SUMMARY_COLUMNS = [
    "日期", "产品", "广告单量", "总发出单量", "真实发货率", "总支付单量", "售前退款率",
    "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功", "签收率", "退回率",
    "复购率", "消耗", "对标揽件单量签收率", "对标揽件单量退回率", "对标揽件单量复购率",
    "平均客单", "客单", "复购", "备货单量", "售前退款单量", "揽件前退款单量",
]
ARRIVAL_ADVERT_COLUMNS = [
    "日期", "产品", "广告单量", "总发出单量", "真实发货率", "总支付单量", "售前退款率",
    "揽件前退款率", "在线支付已完成", "已发货", "售后中", "退回成功", "签收率", "退回率",
    "消耗", "对标揽件单量签收率", "对标揽件单量退回率", "平均客单", "客单", "备货单量",
    "售前退款单量", "揽件前退款单量",
]

RATE_COLUMNS = [
    "真实发货率", "签收率", "退回率", "复购率", "对标支付签收率", "站内订单占比",
    "优惠订单占比", "对标揽件单量签收率", "对标揽件单量退回率", "对标揽件单量复购率",
    "售前退款率", "揽件前退款率",
]

Python 3.7.0 | pandas 0.23.4


## 1. 读取订单并筛选直播商品 / 白名单达人

In [2]:
df = pd.read_csv(ORDER_CSV, encoding="gbk")

# 金额 / 数量列统一转数值（旧版 pandas 可能把它们读成文本，直接运算会报错）
for _col in NUMERIC_COLUMNS:
    if _col in df.columns:
        df[_col] = to_number(df[_col], _col)

# 只保留有「直播」流量体裁的商品
live_goods_ids = df.loc[df["流量体裁"] == "直播", "商品ID"].unique()
df = df[df["商品ID"].isin(live_goods_ids)]

# 白名单之外的达人（仅提示，不参与统计）
other_influencers = (
    df.loc[~df["达人ID"].isin(INFLUENCER_IDS), ["达人ID", "达人昵称"]].drop_duplicates()
)
print(f"直播商品 ID 共 {len(live_goods_ids)} 个")
print("白名单外的达人：")
print(other_influencers if len(other_influencers) else "（无）")

df = df[df["达人ID"].isin(INFLUENCER_IDS)].copy()
print(f"命中白名单的订单行：{len(df):,}")

D:\soft\Anaconda\AN\lib\site-packages\IPython\core\interactiveshell.py:2785: DtypeWarning: Columns (9,11,59) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


直播商品 ID 共 8 个
白名单外的达人：
（无）
命中白名单的订单行：50,455


## 2. 基础字段清洗（时间 / 商家编码 / 两种销售金额口径）

In [3]:
df["订单提交时间"] = df["订单提交时间"].str.lstrip()
df["商家编码"] = df["商家编码"].str[1:]
df["订单提交时间"] = pd.to_datetime(df["订单提交时间"], format="%Y-%m-%d %H:%M:%S")
df["订单提交时间"] = df["订单提交时间"].dt.strftime("%Y-%m-%d")

# 全量版口径：订单应付金额 + 平台实际承担优惠 + 运费
df["销售金额"] = df["订单应付金额"] + df["平台实际承担优惠金额"] + df["运费"]
# 到货版口径：商品数量 × 商品单价 + 运费
df["到货销售金额"] = df["商品数量"] * df["商品单价"] + df["运费"]

## 3. 关联商品编码，导出缺码表

In [4]:
sku = pd.read_excel(SKU_XLSX).rename(columns={"商品名": "产品名称", "商品编码": "商家编码"})
sku = sku[["商家编码", "产品名称"]]
df = df.merge(sku, on="商家编码", how="left").reset_index(drop=True)

lack_code = df.loc[df["产品名称"].isnull(), "商家编码"].value_counts()
lack_code.to_excel(LACK_FILE)
print(f"缺码 {len(lack_code)} 个，已导出 {LACK_FILE}")

缺码 132 个，已导出 缺码表.xlsx


## 4. 合并包裹状态（保留原始状态，供到货口径使用）

In [5]:
df2 = pd.read_excel(PACKAGE_XLSX, converters={"订单编号": str})
df2 = df2.rename(columns={"订单编号": "主订单编号"})
df2 = df2[["主订单编号", "包裹状态", "揽件时间"]]
df2["主订单编号"] = df2["主订单编号"].astype(str)

# 记住原始包裹状态：到货口径 = 原始状态为「已签收」
df2["包裹状态_原始"] = df2["包裹状态"]
df2["包裹状态"] = df2["包裹状态"].replace(PACKAGE_TO_DONE)

# 同一主订单多条运单时取第一条
df2 = df2.drop_duplicates(subset="主订单编号", keep="first")

# 兼容带制表符前缀的订单号
if "\t" in str(df2.loc[0, "主订单编号"]):
    df2["主订单编号"] = df2["主订单编号"].str[:19]

df["主订单编号"] = df["主订单编号"].astype(str)
df3 = df.merge(df2, on="主订单编号", how="left")
df3["主订单编号"] = df3["主订单编号"].astype(str)
df3["子订单编号"] = df3["子订单编号"].astype(str)

# 包裹已完成 + 售后正常 → 订单状态记为已完成
df3.loc[
    df3["包裹状态"].eq("已完成1") & df3["售后状态"].isin({"-", "售后关闭", "换货成功"}),
    "订单状态",
] = "已完成"

print(df3["包裹状态"].value_counts(dropna=False))

已完成1    34294
NaN      8208
运输中      3882
已发货      3223
已退回       642
已取消       205
已揽收         1
Name: 包裹状态, dtype: int64


## 5. 剔除短视频渠道订单，输出检查表

In [6]:
df3 = df3[df3["广告渠道"] != "短视频"].copy()
print(df3["广告渠道"].value_counts(dropna=False))

if WRITE_CHECK:
    # 去掉合并过程中新增的辅助列，保持与原「检查.xlsx」一致
    df3.drop(columns=["包裹状态_原始", "到货销售金额"]).to_excel(CHECK_FILE, index=False)
    print(f"已导出 {CHECK_FILE}")

无    50455
Name: 广告渠道, dtype: int64
已导出 检查.xlsx


## 6. 切分到货订单（已签收）

In [7]:
df_arrival = df3[df3["包裹状态_原始"] == ARRIVAL_RAW_STATUS].copy()
print(f"全量订单行：{len(df3):,} | 到货（已签收）行：{len(df_arrival):,}")

全量订单行：50,455 | 到货（已签收）行：31,808


## 7. 指标计算（两个口径共用）

原代码里 6 处逐行 `for` 循环 + `df.loc[i, ...]` 赋值，全部改为整列布尔掩码。

In [8]:
def add_metrics(df_in, sales_col="销售金额", with_extras=True):
    """补齐单量、状态、时长、复购等指标。

    参数：
        sales_col   —— 销售金额取值列（全量口径 / 到货口径不同）
        with_extras —— 是否计算时长、站内订单、优惠券等扩展指标
    """
    out = df_in.copy()

    # ① 订单数先按商品数量计（后续重算）
    out["订单数"] = out["商品数量"]

    # ② 剔除未支付订单：订单状态=已关闭 且 售后状态=-
    out = out[~((out["订单状态"] == "已关闭") & (out["售后状态"] == "-"))].reset_index(drop=True)

    # ③ 支付 / 退款 / 备货单量
    out["总支付单量"] = out["订单数"]
    out["售前退款单量"] = np.where(
        out["包裹状态"].isna() & out["售后状态"].eq("退款成功"), out["订单数"], 0
    )
    out["揽件前退款单量"] = np.where(
        out["揽件时间"].isna() & out["售后状态"].eq("退款成功"), out["订单数"], 0
    )
    out["备货单量"] = np.where(
        out["包裹状态"].isna() & out["订单状态"].eq("待发货"), out["订单数"], 0
    )

    # ④ 时长 / 站内订单 / 优惠券（仅全量口径需要）
    if with_extras:
        out["发货时长（小时）"] = (
            pd.to_datetime(out["发货时间"]) - pd.to_datetime(out["支付完成时间"])
        ).dt.total_seconds() / 3600
        out["揽件时长（小时）"] = (
            pd.to_datetime(out["揽件时间"]) - pd.to_datetime(out["支付完成时间"])
        ).dt.total_seconds() / 3600
        out["优惠券单量"] = np.where(out["平台优惠"].ne("-"), out["订单数"], 0)
        out["站内订单"] = np.where(
            out["APP渠道"].isin({"抖音", "抖音极速版", "抖音商城版"}), out["订单数"], 0
        )

    # ⑤ 销售金额：填充空值；订单数为 0 时归零
    out["销售金额"] = out[sales_col].fillna(0)
    out["订单数"] = out["总支付单量"] - out["售前退款单量"] - out["备货单量"]
    out["揽件订单数"] = out["总支付单量"] - out["揽件前退款单量"] - out["备货单量"]
    out.loc[out["订单数"] == 0, "销售金额"] = 0

    # ⑥ 状态指标（原 if/elif 链的等价向量化）
    done_after_sale = out["售后状态"].isin(DONE_AFTER_SALE)
    paid_done = out["订单状态"].eq("已完成") & done_after_sale
    refunded = out["售后状态"].eq("退款成功")
    shipped = out["订单状态"].eq("已发货") & done_after_sale

    out["在线支付已完成"] = np.where(paid_done, out["订单数"], 0)
    out["退回成功"] = np.where(refunded, out["订单数"], 0)
    out["已发货"] = np.where(shipped, out["订单数"], 0)
    out["售后中"] = np.where(
        ~(paid_done | refunded | shipped) & ~out["售后状态"].isin(AFTER_SALE_IN_PROGRESS_EXCLUDE),
        out["订单数"],
        0,
    )

    # ⑦ 复购 / 广告
    out["复购"] = np.where(out["广告渠道"].eq("无"), out["订单数"], 0)
    out["广告"] = np.where(out["广告渠道"].isin({"直播", "商品卡"}), out["订单数"], 0)
    return out

## 8. 汇总表生成（参数化，两个口径共用）

In [9]:
# 广告消耗表
df_consume = pd.read_excel(CONSUME_XLSX)[["日期", "产品", "消耗", "转化数"]]
_tmp = df_consume["日期"].str.extract(r"(\d+).*?(\d+).*?(\d+)")
df_consume["订单提交时间"] = _tmp[0] + "-" + _tmp[1] + "-" + _tmp[2]
df_consume = df_consume.drop(columns=["日期"])
df_consume = df_consume.groupby("订单提交时间", as_index=False)[["消耗", "转化数"]].sum()
df_consume["订单提交时间"] = pd.to_datetime(
    df_consume["订单提交时间"], format="%Y-%m-%d"
).dt.strftime("%Y-%m-%d")


def _pivot(df_in, value_cols):
    return pd.pivot_table(
        df_in, index=["订单提交时间"], values=value_cols, aggfunc=np.sum
    ).reset_index()


def _add_rates(frame, full_metrics):
    """计算转化/签收/退款等比率（未格式化）。"""
    frame["真实发货率"] = frame["订单数"] / frame["转化数"]
    frame["签收率"] = frame["在线支付已完成"] / frame["转化数"]
    frame["退回率"] = frame["退回成功"] / frame["转化数"]
    frame["转化成本"] = frame["消耗"] / frame["转化数"]
    frame["平均客单"] = frame["销售金额"] / frame["订单数"]
    frame["对标揽件单量签收率"] = frame["在线支付已完成"] / frame["揽件订单数"]
    frame["对标揽件单量退回率"] = frame["退回成功"] / frame["揽件订单数"]
    frame["对标揽件单量复购率"] = frame["复购"] / frame["揽件订单数"]
    frame["售前退款率"] = frame["售前退款单量"] / frame["总支付单量"]
    frame["揽件前退款率"] = frame["揽件前退款单量"] / frame["总支付单量"]

    if full_metrics:
        frame["对标支付签收率"] = frame["在线支付已完成"] / frame["总支付单量"]
        frame["发货时长"] = frame["发货时长（小时）"] / frame["订单数"]
        frame["揽件时长"] = frame["揽件时长（小时）"] / frame["揽件订单数"]
        frame["站内订单占比"] = frame["站内订单"] / frame["总支付单量"]
        frame["优惠订单占比"] = frame["优惠券单量"] / frame["总支付单量"]
    return frame


def _format_rates(frame):
    for col in RATE_COLUMNS:
        if col in frame.columns:
            frame[col] = frame[col].apply(lambda x: format(x, ".2%"))
    for col in ("转化成本", "平均客单"):
        if col in frame.columns:
            frame[col] = frame[col].round(2)
    return frame


def build_tables(df_in, full_metrics, 复购率_by_paid, overwrite_复购,
                 summary_columns, advert_columns):
    """生成（汇总, 广告）两张表。"""
    value_cols = BASE_PIVOT_VALUES + (EXTRA_PIVOT_VALUES if full_metrics else [])
    result_cols = EXTRA_RESULT_COLUMNS if full_metrics else BASE_RESULT_COLUMNS

    # ---- 汇总表 ----
    result = _pivot(df_in, value_cols)[result_cols]
    final = result.merge(df_consume, on=["订单提交时间"], how="left")
    final = _add_rates(final, full_metrics)

    if 复购率_by_paid:
        final["复购率"] = (final["总支付单量"] - final["转化数"]) / final["转化数"]
    else:
        final["复购率"] = final["复购"] / final["转化数"]

    final = _format_rates(final)
    if overwrite_复购:
        # 原「标准版」在最后把复购替换为 总支付单量 − 转化数
        final["复购"] = final["总支付单量"] - final["转化数"]

    final = final.rename(columns=RENAME_COLUMNS)
    final["产品"] = "直播产品"
    final = final[summary_columns]

    # ---- 广告表（只看直播 / 商品卡流量） ----
    df_advert = df_in[df_in["流量体裁"].isin({"直播", "商品卡"})]
    result_advert = _pivot(df_advert, value_cols)
    final_advert = result_advert.merge(df_consume, on=["订单提交时间"], how="left")
    final_advert = _format_rates(_add_rates(final_advert, full_metrics))
    final_advert = final_advert.rename(columns=RENAME_COLUMNS)
    final_advert["产品"] = "直播产品"
    final_advert = final_advert[advert_columns]

    return final, final_advert

## 9. 生成「直播汇总.xlsx」（全量口径）

In [10]:
df_full = add_metrics(df3, sales_col="销售金额", with_extras=True)
final, final_advert = build_tables(
    df_full,
    full_metrics=True,
    复购率_by_paid=True,
    overwrite_复购=True,
    summary_columns=FULL_SUMMARY_COLUMNS,
    advert_columns=FULL_ADVERT_COLUMNS,
)

with pd.ExcelWriter(SUMMARY_FILE, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="汇总")          # 与原文件一致：汇总带索引
    final_advert.to_excel(writer, sheet_name="广告", index=False)
print(f"已导出 {SUMMARY_FILE}：汇总 {final.shape[0]} 行，广告 {final_advert.shape[0]} 行")

已导出 直播汇总.xlsx：汇总 21 行，广告 21 行


## 10. 生成「直播汇总-到货订单.xlsx」（到货 / 已签收口径）

In [11]:
df_arrival_metrics = add_metrics(
    df_arrival, sales_col="到货销售金额", with_extras=False
)
final_arr, final_advert_arr = build_tables(
    df_arrival_metrics,
    full_metrics=False,
    复购率_by_paid=False,
    overwrite_复购=False,
    summary_columns=ARRIVAL_SUMMARY_COLUMNS,
    advert_columns=ARRIVAL_ADVERT_COLUMNS,
)

with pd.ExcelWriter(SUMMARY_ARRIVAL_FILE, engine="openpyxl") as writer:
    final_arr.to_excel(writer, sheet_name="汇总")
    final_advert_arr.to_excel(writer, sheet_name="广告", index=False)
print(f"已导出 {SUMMARY_ARRIVAL_FILE}：汇总 {final_arr.shape[0]} 行，广告 {final_advert_arr.shape[0]} 行")

已导出 直播汇总-到货订单.xlsx：汇总 21 行，广告 21 行


## 11. 结果预览

In [12]:
final.head()

,日期,产品,广告单量,总发出单量,真实发货率,总支付单量,售前退款率,揽件前退款率,在线支付已完成,已发货,售后中,退回成功,签收率,退回率,复购率,消耗,发货时长,揽件时长,优惠订单占比,站内订单占比,对标支付签收率,对标揽件单量签收率,对标揽件单量退回率,对标揽件单量复购率,平均客单,客单,复购,备货单量,售前退款单量,揽件前退款单量,发货时长（小时）,揽件时长（小时）,站内订单,优惠券单量
0,2026-10-16,直播产品,1441.0,1393,96.67%,1502,7.26%,13.65%,1115,0,0,278,77.38%,19.29%,4.23%,39149.95,0.124482,7.286517,30.89%,76.36%,74.23%,85.97%,21.43%,107.40%,59.68,83130.7,61.0,0,109,205,173.403056,9450.612222,1147,464
1,2026-10-17,直播产品,1444.0,1372,95.01%,1486,7.67%,13.66%,1092,0,2,278,75.62%,19.25%,2.91%,38958.66,0.138808,7.557794,30.96%,66.08%,73.49%,85.11%,21.67%,106.94%,60.36,82812.8,42.0,0,114,203,190.444722,9696.649722,982,460
2,2026-10-18,直播产品,1258.0,1238,98.41%,1311,5.57%,9.69%,1019,0,1,218,81.00%,17.33%,4.21%,34395.28,0.145099,7.330507,31.96%,66.90%,77.73%,86.06%,18.41%,104.56%,59.81,74046.2,53.0,0,73,127,179.632222,8679.320278,877,419
3,2026-10-19,直播产品,1186.0,1126,94.94%,1214,7.25%,13.18%,877,0,5,244,73.95%,20.57%,2.36%,30995.51,0.142279,8.959804,56.34%,69.52%,72.24%,83.21%,23.15%,106.83%,57.24,64447.4,28.0,0,88,160,160.206667,9443.633333,844,684
4,2026-10-20,直播产品,1565.0,1505,96.17%,1602,6.05%,12.48%,1201,0,7,297,76.74%,18.98%,2.36%,43227.31,0.150290,6.947656,84.33%,72.78%,74.97%,85.66%,21.18%,107.35%,54.77,82434.5,37.0,0,97,200,226.186111,9740.613056,1166,1351


In [13]:
final_arr.head()

,日期,产品,广告单量,总发出单量,真实发货率,总支付单量,售前退款率,揽件前退款率,在线支付已完成,已发货,售后中,退回成功,签收率,退回率,复购率,消耗,对标揽件单量签收率,对标揽件单量退回率,对标揽件单量复购率,平均客单,客单,复购,备货单量,售前退款单量,揽件前退款单量
0,2026-10-16,直播产品,1441.0,1263,87.65%,1263,0.00%,0.00%,1115,0,0,148,77.38%,10.27%,87.65%,39149.95,88.28%,11.72%,100.00%,118.94,150223.7,1263,0,0,0
1,2026-10-17,直播产品,1444.0,1251,86.63%,1251,0.00%,0.00%,1088,0,2,161,75.35%,11.15%,86.63%,38958.66,86.97%,12.87%,100.00%,120.23,150404.9,1251,0,0,0
2,2026-10-18,直播产品,1258.0,1161,92.29%,1161,0.00%,0.00%,1015,0,1,145,80.68%,11.53%,92.29%,34395.28,87.42%,12.49%,100.00%,118.51,137593.9,1161,0,0,0
3,2026-10-19,直播产品,1186.0,1024,86.34%,1024,0.00%,0.00%,873,0,4,147,73.61%,12.39%,86.34%,30995.51,85.25%,14.36%,100.00%,117.09,119897.6,1024,0,0,0
4,2026-10-20,直播产品,1565.0,1375,87.86%,1375,0.00%,0.00%,1198,0,6,171,76.55%,10.93%,87.86%,43227.31,87.13%,12.44%,100.00%,115.99,159492.5,1375,0,0,0
